<a href="https://colab.research.google.com/github/kannanrk28/Capstone_Masai_FinalProject_Kannan/blob/main/part4/Part4_LLM_Powered_Feature.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ====================================================================================================
# Part 4 - Track C: Model Prediction Explanation Pipeline
# Cell 1 - Imports
# ====================================================================================================

!pip install -q jsonschema joblib

import os
import re
import json
import joblib
import requests
import pandas as pd

from jsonschema import validate
from jsonschema.exceptions import ValidationError


In [ ]:
# ====================================================================================================
# Cell 2 - Secure API configuration
# ====================================================================================================

from google.colab import userdata

api_key = userdata.get("LLM_API_KEY")
api_url = userdata.get("LLM_API_URL")
llm_model = userdata.get("LLM_MODEL")

if not api_key:
    raise ValueError("LLM_API_KEY is missing from Colab Secrets.")

if not api_url:
    raise ValueError("LLM_API_URL is missing from Colab Secrets.")

if not llm_model:
    raise ValueError("LLM_MODEL is missing from Colab Secrets.")

print("LLM API configuration loaded successfully.")

In [ ]:
# ====================================================================================================
# Cell 3 - Download and load best_model.pkl
# ====================================================================================================

MODEL_RAW_URL = (
    "https://raw.githubusercontent.com/"
    "kannanrk28/Capstone_Masai_FinalProject_Kannan/"
    "197f93a5a8e7e489351ca72f72f1746fc26d40d6/"
    "part4/best_model.pkl"
)

MODEL_LOCAL_PATH = "best_model.pkl"

best_model = None

try:
    response = requests.get(
        MODEL_RAW_URL,
        timeout=60
    )

    response.raise_for_status()

    with open(MODEL_LOCAL_PATH, "wb") as model_file:
        model_file.write(response.content)

    best_model = joblib.load(MODEL_LOCAL_PATH)

    print("Best model downloaded and loaded successfully.")
    print("Model type:", type(best_model))

except requests.exceptions.RequestException as error:
    print("Model download failed:", error)

except Exception as error:
    print("Model loading failed:", error)

In [ ]:
# ====================================================================================================
# Cell 4 - Zero-shot system prompt
# ====================================================================================================

system_prompt = """
You are an AI assistant that explains machine learning classification predictions.

Your task is to generate a structured explanation for a movie revenue prediction.

You will receive:
1. A set of movie feature values.
2. The predicted class produced by the machine learning model.
3. The predicted probability for that class.

Return ONLY valid JSON using the following schema:

{
  "prediction_label": "string",
  "confidence_level": "low | medium | high",
  "top_reason": "string",
  "second_reason": "string",
  "next_step": "string"
}

Instructions:
- Return only valid JSON without Markdown or additional text.
- Do not invent or modify feature values.
- Do not claim that a feature caused the prediction.
- Describe only features that may have influenced the prediction.
- Base the explanation only on the provided feature values and prediction.
- Determine confidence_level using the predicted class probability:
    - high: probability >= 0.80
    - medium: probability >= 0.60 and < 0.80
    - low: probability < 0.60
- Keep each explanation concise and easy to understand.
""".strip()

print(system_prompt)

In [ ]:
# ====================================================================================================
# Cell 5 - JSON schema and fallback
# ====================================================================================================

explanation_schema = {
    "type": "object",
    "properties": {
        "prediction_label": {
            "type": "string"
        },
        "confidence_level": {
            "type": "string",
            "enum": ["low", "medium", "high"]
        },
        "top_reason": {
            "type": "string"
        },
        "second_reason": {
            "type": "string"
        },
        "next_step": {
            "type": "string"
        }
    },
    "required": [
        "prediction_label",
        "confidence_level",
        "top_reason",
        "second_reason",
        "next_step"
    ],
    "additionalProperties": False
}

fallback_explanation = {
    "prediction_label": None,
    "confidence_level": None,
    "top_reason": None,
    "second_reason": None,
    "next_step": None
}

print("JSON schema and fallback response created.")

In [ ]:
# ====================================================================================================
# Cell 6 - Reusable helper functions
# ====================================================================================================

def call_llm(
    system_prompt,
    user_prompt,
    temperature=0.0,
    max_tokens=512
):
    """
    Send a request to the LLM API and return the assistant response text.
    """

    payload = {
        "model": llm_model,
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    try:
        response = requests.post(
            api_url,
            headers=headers,
            json=payload,
            timeout=60
        )

    except requests.exceptions.RequestException as error:
        print("LLM request error:", error)
        return None

    if response.status_code != 200:
        print("LLM request failed.")
        print("Status code:", response.status_code)
        print("Response:", response.text)
        return None

    try:
        return response.json()["choices"][0]["message"]["content"]

    except (KeyError, IndexError, TypeError, ValueError) as error:
        print("Unexpected LLM response structure:", error)
        return None


def has_pii(text):
    """
    Detect email addresses and common 10-digit phone-number formats.
    """

    email_pattern = (
        r"[a-zA-Z0-9_.+-]+@"
        r"[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+"
    )

    phone_pattern = (
        r"\b\d{10}\b|"
        r"\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b"
    )

    return bool(
        re.search(email_pattern, text) or
        re.search(phone_pattern, text)
    )


def safe_call_llm(
    system_prompt,
    user_prompt,
    temperature=0.0,
    max_tokens=512
):
    """
    Apply the PII guardrail before calling the LLM.
    """

    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        return None, "Block"

    response = call_llm(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        temperature=temperature,
        max_tokens=max_tokens
    )

    return response, "Pass"


def encode_record(features):
    """
    Convert one feature dictionary into a one-row DataFrame and
    align it with the columns used during model training.
    """

    if best_model is None:
        raise ValueError("The best model has not been loaded.")

    record = pd.DataFrame([features])

    if hasattr(best_model, "feature_names_in_"):
        expected_columns = list(best_model.feature_names_in_)

        record = record.reindex(
            columns=expected_columns,
            fill_value=0
        )

    return record


def build_user_prompt(
    feature_values,
    predicted_class,
    predicted_probability
):
    """
    Create the structured user prompt for Track C.
    """

    prediction_label = (
        "High Revenue"
        if predicted_class == 1
        else "Low Revenue"
    )

    return f"""
Movie Features:
{json.dumps(feature_values, indent=2)}

Predicted Class:
{predicted_class}

Prediction Label:
{prediction_label}

Predicted Probability:
{predicted_probability:.6f}

Generate the structured JSON explanation.
""".strip()


def clean_json_text(raw_response):
    """
    Remove whitespace and optional Markdown code fences.
    """

    cleaned = raw_response.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]

    elif cleaned.startswith("```"):
        cleaned = cleaned[3:]

    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]

    return cleaned.strip()


def parse_and_validate(raw_response):
    """
    Parse the LLM response and validate it against the JSON schema.
    """

    if raw_response is None:
        print("Validation error: no LLM response.")
        return fallback_explanation.copy(), "Failed"

    cleaned_response = clean_json_text(raw_response)

    try:
        parsed_response = json.loads(cleaned_response)

    except json.JSONDecodeError as error:
        print("JSON parsing error:", error)
        return fallback_explanation.copy(), "JSON Failed"

    try:
        validate(
            instance=parsed_response,
            schema=explanation_schema
        )

    except ValidationError as error:
        print("Schema validation error:", error.message)
        return fallback_explanation.copy(), "Schema Failed"

    return parsed_response, "Passed"


print("Helper functions created successfully.")

In [ ]:
# ====================================================================================================
# Cell 7 - Three hand-crafted feature vectors
# ====================================================================================================

test_inputs = [
    {
        "budget": 150_000_000,
        "runtime": 130,
        "release_year": 2020,
        "release_month": 7,
        "Action": 1,
        "Adventure": 1,
        "Animation": 0,
        "Comedy": 0,
        "Crime": 0,
        "Documentary": 0,
        "Drama": 0,
        "Family": 0,
        "Fantasy": 1,
        "Foreign": 0,
        "History": 0,
        "Horror": 0,
        "Music": 0,
        "Mystery": 0,
        "Romance": 0,
        "Science Fiction": 1,
        "TV Movie": 0,
        "Thriller": 1,
        "War": 0,
        "Western": 0
    },
    {
        "budget": 10_000_000,
        "runtime": 95,
        "release_year": 2018,
        "release_month": 2,
        "Action": 0,
        "Adventure": 0,
        "Animation": 0,
        "Comedy": 1,
        "Crime": 0,
        "Documentary": 0,
        "Drama": 1,
        "Family": 0,
        "Fantasy": 0,
        "Foreign": 0,
        "History": 0,
        "Horror": 0,
        "Music": 0,
        "Mystery": 0,
        "Romance": 1,
        "Science Fiction": 0,
        "TV Movie": 0,
        "Thriller": 0,
        "War": 0,
        "Western": 0
    },
    {
        "budget": 45_000_000,
        "runtime": 110,
        "release_year": 2015,
        "release_month": 10,
        "Action": 0,
        "Adventure": 0,
        "Animation": 0,
        "Comedy": 0,
        "Crime": 1,
        "Documentary": 0,
        "Drama": 1,
        "Family": 0,
        "Fantasy": 0,
        "Foreign": 0,
        "History": 0,
        "Horror": 0,
        "Music": 0,
        "Mystery": 1,
        "Romance": 0,
        "Science Fiction": 0,
        "TV Movie": 0,
        "Thriller": 1,
        "War": 0,
        "Western": 0
    }
]

print("Number of test inputs:", len(test_inputs))

In [ ]:
# ====================================================================================================
# Cell 8 - Simple LLM API test
# ====================================================================================================

test_response, guardrail_result = safe_call_llm(
    system_prompt="You are a helpful assistant.",
    user_prompt="Reply with only the word: hello",
    temperature=0.0,
    max_tokens=20
)

print("Guardrail result:", guardrail_result)
print("LLM response:", test_response)

In [ ]:
# ====================================================================================================
# Cell 9 - PII guardrail demonstration
# ====================================================================================================

# Test 1: Contains PII and should be blocked
pii_input = """
Customer email is john.doe@gmail.com.
Please explain the prediction.
"""

blocked_response, blocked_status = safe_call_llm(
    system_prompt=system_prompt,
    user_prompt=pii_input,
    temperature=0.0
)

print("\nPII Test")
print("Guardrail status:", blocked_status)
print("Response:", blocked_response)


# Test 2: Contains no PII and should proceed
safe_input = """
Movie budget is 150000000.
Runtime is 130 minutes.
Predicted class is High Revenue.
Predicted probability is 0.95.
Generate the structured JSON explanation.
"""

allowed_response, allowed_status = safe_call_llm(
    system_prompt=system_prompt,
    user_prompt=safe_input,
    temperature=0.0
)

print("\nNon-PII Test")
print("Guardrail status:", allowed_status)
print("Response:", allowed_response)

In [ ]:
# ====================================================================================================
# Cell 10 - Structured output handling
# ====================================================================================================

validation_results = []

if best_model is None:
    raise ValueError("Model is unavailable. Run the model-loading cell first.")

for input_number, feature_values in enumerate(test_inputs, start=1):

    print("\n" + "=" * 90)
    print(f"INPUT {input_number}")
    print("=" * 90)

    encoded_record = encode_record(feature_values)

    predicted_class = int(
        best_model.predict(encoded_record)[0]
    )

    class_probabilities = best_model.predict_proba(
        encoded_record
    )[0]

    predicted_probability = float(
        class_probabilities[predicted_class]
    )

    user_prompt = build_user_prompt(
        feature_values=feature_values,
        predicted_class=predicted_class,
        predicted_probability=predicted_probability
    )

    raw_response, guardrail_result = safe_call_llm(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        temperature=0.0,
        max_tokens=512
    )

    explanation, validation_status = parse_and_validate(
        raw_response
    )

    print("\nFeature Values:")
    print(json.dumps(feature_values, indent=2))

    print("\nPredicted Class:")
    print(predicted_class)

    print("\nPredicted Probability:")
    print(f"{predicted_probability:.6f}")

    print("\nRaw LLM Response:")
    print(raw_response)

    print("\nValidated Explanation:")
    print(json.dumps(explanation, indent=2))

    print("\nValidation Outcome:")
    print(validation_status)

    print("\nGuardrail Result:")
    print(guardrail_result)

    validation_results.append({
        "Feature Input": f"Input {input_number}",
        "Predicted Class": predicted_class,
        "Probability": round(predicted_probability, 4),
        "Explanation JSON": explanation,
        "Validation Status": validation_status,
        "Guardrail Result": guardrail_result
    })


validation_results_df = pd.DataFrame(validation_results)

print("\nValidation Results DataFrame:")
display(validation_results_df)

In [ ]:
# ====================================================================================================
# Cell 11 - Temperature 0 vs 0.7 comparison
# ====================================================================================================

temperature_results = []

for input_number, feature_values in enumerate(test_inputs, start=1):

    encoded_record = encode_record(feature_values)

    predicted_class = int(
        best_model.predict(encoded_record)[0]
    )

    class_probabilities = best_model.predict_proba(
        encoded_record
    )[0]

    predicted_probability = float(
        class_probabilities[predicted_class]
    )

    user_prompt = build_user_prompt(
        feature_values=feature_values,
        predicted_class=predicted_class,
        predicted_probability=predicted_probability
    )

    output_temp_0, status_temp_0 = safe_call_llm(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        temperature=0.0,
        max_tokens=512
    )

    output_temp_07, status_temp_07 = safe_call_llm(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        temperature=0.7,
        max_tokens=512
    )

    print("\n" + "=" * 90)
    print(f"INPUT {input_number}")
    print("=" * 90)

    print("\nOutput at temperature = 0:")
    print(output_temp_0)

    print("\nOutput at temperature = 0.7:")
    print(output_temp_07)

    temperature_results.append({
        "Input": f"Input {input_number}",
        "Output at temp=0": output_temp_0,
        "Output at temp=0.7": output_temp_07,
        "Temp 0 Guardrail": status_temp_0,
        "Temp 0.7 Guardrail": status_temp_07
    })


temperature_results_df = pd.DataFrame(temperature_results)

print("\nTemperature Comparison DataFrame:")
display(temperature_results_df)

In [ ]:
# ====================================================================================================
# Cell 12 - End-to-end summary table
# ====================================================================================================

end_to_end_results = []

for item in validation_results:
    end_to_end_results.append({
        "Input": item["Feature Input"],
        "LLM Output": json.dumps(item["Explanation JSON"]),
        "Valid JSON": item["Validation Status"],
        "Pass/Block": item["Guardrail Result"]
    })

end_to_end_df = pd.DataFrame(end_to_end_results)

print("End-to-End Summary:")
display(end_to_end_df)

In [ ]:
# ====================================================================================================
# Cell 13 - Generate README Markdown tables
# ====================================================================================================

readme_validation_table = validation_results_df[
    [
        "Feature Input",
        "Predicted Class",
        "Probability",
        "Explanation JSON",
        "Validation Status"
    ]
].copy()

readme_validation_table["Explanation JSON"] = (
    readme_validation_table["Explanation JSON"]
    .apply(json.dumps)
)

print("Structured Output Validation Table:\n")
print(readme_validation_table.to_markdown(index=False))


readme_end_to_end_table = end_to_end_df.copy()

print("\n\nEnd-to-End Table:\n")
print(readme_end_to_end_table.to_markdown(index=False))